# Folding story: a new picture at every stage

One printed square that tells a short story as you fold it into a crane. Each stage of folding shows a different picture:

| Stage | Example |
|---|---|
| Flat sheet | a field of leaves |
| Preliminary base | a flower bud |
| Bird base | a half-open flower |
| Crane | a flower in full bloom |

This extends [`illusions.ipynb`](illusions.ipynb), which makes one picture flat and two on the folded crane. Here the folded views come from three different models: the crane and the two bases you pass through on the way to it.

**The catch:** every picture lives on the same sheet, so pictures compete for pixels in two ways.

- **Against the flat sheet.** The flat view shows every pixel, so each stage's picture has to hide in the flat picture. How hard that is depends on how much folding scrambles the stage's visible paper (table below).
- **Between consecutive stages.** Folding keeps much of the outside on the outside, so neighboring stages show a lot of the same paper. With a crane, up to about half of a stage's visible paper also shows at another stage. Those pixels are painted once for both pictures, so consecutive scenes should be able to share strokes. That's why the default story is a transformation (a bud opening) rather than unrelated pictures. An early test with the digits 1, 2, 3 made the 2 and the 3 blur into each other for this reason.

| Stage | Pieces of paper on the visible side | Hiding the picture when flat |
|---|---|---|
| Preliminary base | 1 (an uncut quarter of the sheet, turned 135°) | hard: the picture *is* a corner of the flat picture |
| Bird base | 2 | medium |
| Crane | about 10 | easy, as in trial 1 |

## Setup

Clones [rath-hy/doodles](https://github.com/rath-hy/doodles/tree/main/2026-10-31-fold-map) on Colab, or uses the local files if you run this from that folder.

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "2026-10-31-fold-map"
if not os.path.exists("story.py"):
    if not os.path.exists("doodles"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/doodles.git"], check=True)
    os.chdir(os.path.join("doodles", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "directions", "decal", "illusion", "story"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel, PALETTE
from illusion import printable
from story import STAGES, least_shared_sides, plot_story, shared_paper, stage_views

## The stages

The bases' crease patterns and folded states are in `stages/`, made by `make_stages.py` with Flat-Folder's solver (the same tool that gave us the crane). Their creases match the crane's, so all three share one sheet.

Each stage can be held either way round. We pick the sides so that no stage shares too much of its visible paper with another stage. For the crane, that leaves the finished crane with no shared paper, while the two bases share some with each other. Each stage is shown upright: the preliminary base with its closed point up, the bird base with its wing flaps up, the crane as Flat-Folder draws it.

In the maps below, color is the paper a stage shows, on the printed side; everything else is hidden at that stage.

In [ ]:
models = [FoldedModel(cp, state) for _, cp, state, _ in STAGES]
SIDES = least_shared_sides(models)
names = [name for name, _, _, _ in STAGES]

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 5))
seen = []
for ax, m, side, name in zip(axes, models, SIDES, names):
    _, side2, _, _ = m.sheet_visibility(300)
    shown = side2 == (1 if side == "front" else 2)
    ax.imshow(np.where(shown[..., None], PALETTE[1 if side == "front" else 2], 0.95))
    ax.set_title(f"{name}, {side}: {shown.mean():.0%} of the sheet")
    ax.axis("off")
plt.show()
for name, share in zip(names, shared_paper(models, SIDES)):
    print(f"{name}: {share:.0%} of its visible paper also shows at another stage")

## 1. Digits (no AI)

The simplest version: 0 flat, 1, 2, 3 on the stages, placed so they compete for as few pixels as possible (`python story.py`). It takes about half a minute.

It shows the problem plainly. The 1 on the preliminary base is readable on the flat sheet, because that base shows an uncut quarter of the sheet. Plain digits on white have nothing to hide in.

In [ ]:
from story import plan, render

TEXT = "0123"
views, places, report = plan(TEXT, models)
for k, v in report.items():
    print(f"{k}: {v}")
printed, _ = render(TEXT, models, views, places, 600)
titles = [f"flat: {TEXT[0]}"] + [f"{v.name} ({v.view}): {ch}" for v, ch in zip(views, TEXT[1:])]
plot_story(printed / 255, models, [v.view for v in views], titles);

## 2. Pictures that hide each other (needs a GPU)

This is the sampler from `illusions.ipynb` with more views. At each denoising step, DeepFloyd IF sees the sheet four ways, each with its own prompt: flat, and folded into each stage. Their predictions are mapped back onto the sheet and blended:

- **Pixels hidden at every stage** follow only the flat picture.
- **A pixel shown at some stages** blends the flat prediction with each of those stages' predictions. `FOLD_WEIGHT` sets how much the folded pictures count.

Because every view is generated together, the model can choose content that works in all of them. For example, the bud's petals can double as leaves in the field, and the half-open flower can reuse the bud's strokes.

The preliminary base is the real test. Its picture is a corner of the flat picture, turned 135°. So the flat scene has to contain, in that corner, something that reads as a bud once it's isolated and turned: the rotation trick from Visual Anagrams, applied to one quarter. Expect it to be the weakest stage, and try several seeds.

### Before running

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the license for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece
from huggingface_hub import notebook_login
notebook_login()

### The story

One prompt for the flat sheet, then one per stage, in folding order. Tips:

- **Share a style** across all prompts. Painterly styles hide seams.
- **Make the flat scene busy.** Leaves, twigs, a crowd, a mosaic: texture gives the later pictures somewhere to hide.
- **Tell a transformation.** Consecutive stages share paper, so consecutive scenes should share strokes: a bud opening, a caterpillar becoming a butterfly.
- **Prefer flexible subjects** that can be drawn many ways (flowers, animals) over rigid symbols like digits and letters.
- **Fit each subject to its stage's outline.** The preliminary base is a square diamond, the bird base a tall skinny diamond, the crane a crane.

In [ ]:
STYLE = "a watercolor painting of "
FLAT_PROMPT = STYLE + "a field of green leaves"
STAGE_PROMPTS = [
    STYLE + "a closed pink flower bud, on a plain light background",       # preliminary base
    STYLE + "a half-open pink flower, on a plain light background",        # bird base
    STYLE + "a pink flower in full bloom, on a plain light background",    # crane
]

### Encode the prompts

As in `illusions.ipynb`: the text encoder is loaded onto the GPU, used once, and deleted to save memory. **Rerun this cell whenever you change the prompts.** The first run downloads about 11 GB.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
prompt_embeds, negative_embeds = encoder.encode_prompt([FLAT_PROMPT] + STAGE_PROMPTS)

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("embeddings:", tuple(prompt_embeds.shape))

### Load the image models

Once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

### Generate

- `FOLD_WEIGHT`: how much shown pixels follow the stage pictures (1) vs. the flat one (0). Raise it if a stage's picture is swallowed by the flat scene (the preliminary base is the usual victim).
- `GUIDANCE`: how strongly each stage follows the prompts (stage 1, stage 2).
- `SEED`: try several. Some seeds hide the bud much better than others.

The previews fold the sheet with the decal simulator, independently of the maps used to make it.

In [ ]:
from illusion import generate

FOLD_WEIGHT = 0.6
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

views64, views256 = stage_views(models, SIDES, 64), stage_views(models, SIDES, 256)
sheet64, sheet256 = generate(stage_1, stage_2, views64, views256, (negative_embeds, prompt_embeds),
                             guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=SEED)
titles = ["flat"] + [f"{name} ({side})" for name, side in zip(names, SIDES)]
plot_story(sheet64, models, SIDES, titles);
plot_story(sheet256, models, SIDES, titles);

### Print it

Saves the sheet, and an A4 PDF with two copies (148.5 mm squares, one in the top-left corner and one in the bottom-left), so you get two tries. Print at **actual size** and fold with the **printed side out**.

To check which side of each stage to look at: the preliminary base and bird base are held point up; the side shown is the one in the preview above. If a stage looks wrong, turn it over.

In [ ]:
name = f"story_seed{SEED}"
img = printable(sheet256, 1754)                 # 148.5 mm at 300 dpi
img.save(name + ".png", dpi=(300, 300))
page = Image.new("RGB", (2480, 3508), "white")  # A4 at 300 dpi
page.paste(img, (0, 0))
page.paste(img, (0, 3508 - 1754))
page.save(name + "_a4.pdf", resolution=300)
print("saved", name + ".png", "and", name + "_a4.pdf")
try:
    from google.colab import files
    files.download(name + "_a4.pdf")
except ImportError:
    pass

## Notes

- **The sampler treats all views as equals.** Hiding each picture until its stage comes is something we hope the result does, not something it enforces. If a stage's picture shows on the flat sheet too much, lower `FOLD_WEIGHT` or try another seed.
- **Shared paper sets what's possible.** Pixels that two stages both show can only be one color. Check the shared-paper numbers above when choosing stages and scenes.
- **The preliminary base is the bottleneck.** If it never hides well, swap it for another flat stage of the crane that scrambles the paper more, such as the bird base with its legs narrowed. `make_stages.py` makes new stages from crease lines.
- **Measure it.** Generate many seeds and score each view against its prompt with CLIP, as Visual Anagrams does. That separates luck from settings.

## References

- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.
- A. Lugmayr et al. [RePaint](https://arxiv.org/abs/2201.09865): Inpainting using Denoising Diffusion Probabilistic Models. *CVPR*, 2022.
- H. A. Akitaya, E. D. Demaine, J. S. Ku. [Computing Flat-Folded States](https://erikdemaine.org/papers/FlatFolder_OSME2024/). *OSME*, 2024.